# Capstone — Research Paper: Content Refresh Prioritization using Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FEZEKIL/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

## Abstract
This research investigates whether content-level signals—specifically age, search visibility, and ranking position—can directionalize the prioritization of content refreshes. Using a pseudonymized warehouse of 79 million daily search performance records, we developed a Random Forest model to identify pages at risk of organic traffic decay. While a standard random validation design suggests high accuracy (ROC-AUC 0.772), a more rigorous client-grouped split reveals a significant drop in generalization skill (ROC-AUC 0.609), indicating that site-specific technical signatures drive a large portion of apparent performance. We conclude that machine learning serves as an effective decision-support tool for ranking human content reviews rather than an autonomous scheduling agent.

## 1. Question

**Research Question:** Can we predict future organic traffic decline using trailing content metrics to prioritize manual content refreshes?

**Decision Support:** Content marketing teams currently rely on simple time-based rules (e.g., "refresh every 6 months"). This study aims to replace these static rules with a dynamic, data-driven ranking that identifies pages with high search presence that are showing signs of decay, ensuring limited editorial resources are applied where they have the most impact.

## 2. Data

**Dataset:** FlyRank Internship Warehouse Release (Build v20260703).
**Scale:** ~79 million rows of daily performance records covering ~70 pseudonymized clients and ~520,000 content items.
**Time Windows:** 
- **Feature Window:** February 2026 (aggregated metrics).
- **Label Window:** March 2026 (outcome calculation).
**Exclusions:** We excluded pages with fewer than 5 clicks in the baseline month to ensure stability in the percentage-based decline label. We also excluded new content created during the evaluation window to avoid new-content bias.

## 3. Methodology

**Model:** Random Forest Classifier (100 estimators, depth 10).
**Label Definition:** `is_declining` = 1 if March clicks are < 80% of February clicks (>20% decline).
**Baseline:** A transparent rule-based score: `(days_since_last_update >= 180) * impressions_90d`.
**Features:** Log-transformed impressions, clicks, and sessions; average position; content age; word count; and categorical indicators for content type and search intent.
**Validation Design:** Grouped Client Split (20% holdout). By holding out entire clients (sites), we prevent the model from memorizing site-specific technical characteristics (e.g., internal linking structure or domain authority) that don't generalize to the wider internet.
**Leakage Audit:** Verified that `trend_direction` and outcome-window metrics were strictly excluded from the feature set.

## 4. Results (vs baseline)

The model demonstrates directional skill in identifying decay risk but highlights the critical importance of validation design.

| Validation Design | Metric | Baseline Score | Model Score | Lift |
|---|---|---|---|---|
| Random Split | ROC-AUC | 0.500 | 0.772 | +54% |
| **Grouped Client Split** | **ROC-AUC** | **0.500** | **0.609** | **+22%** |
| Precision @ Top 50 | Precision | 0.700 | 0.560 | -20% |

The 0.163 AUC gap between random and grouped splits reveals significant 'memorization' of client-specific signatures. Furthermore, while the model generalizes well across the whole dataset (higher AUC), the manual baseline remains more effective at identifying the most obvious high-volume refresh candidates (higher Precision@50).

In [ ]:
import IPython.display
IPython.display.SVG("../figures/model_skill_comparison.svg")

## 5. Limitations

**1. Survivorship Bias:** The model only evaluates content that has lived long enough to accumulate search history. It cannot score the refresh opportunity for brand-new pages during their first ~30-60 days.
**2. Intent Blindness:** The dataset contains high-level intent categories (e.g., 'informational') but lacks semantic depth. A page might 'decline' because the search intent for its keyword has fundamentally shifted, which no refresh can fix.
**3. Seasonality:** Monthly windows are sensitive to calendar effects (e.g., February vs. March length and seasonal topics) which may be misclassified as content decay.

## 6. Ranked recommendations

We recommend a **Prioritized Content Review Queue** with the following reason codes:

- **STALE_CONTENT:** Pages not updated in >180 days.
- **STRIKING_DISTANCE:** Pages ranked between 4-10 that have significant impression volume to regain.
- **LOW_CTR_HIGH_VIS:** Pages with above-median impressions but below-median CTR, suggesting meta-data decay.

**Protocol:** Humans should review the top 50 flagged items monthly. Automated updates should be avoided for high-value brand pages or legal/regulatory content.

## 7. Artifacts the paper embeds

This paper builds on the data artifacts generated in `work/notebooks/w07_action_playbook.ipynb` and `work/outputs/playbook_metrics.json`. All underlying code for the validation audit and honest claims can be found in the repository.

## Acknowledgments & Data Credit

This research was built on the **FlyRank ML Internship dataset**. We thank the [FlyRank](https://flyrank.ai) team for providing access to the pseudonymized search intelligence warehouse.

## Reproducibility

- **Repository:** [FEZEKIL/flyrank-ml-internship](https://github.com/FEZEKIL/flyrank-ml-internship)
- **Environment:** Python 3.11, DuckDB, Scikit-learn, Pandas.
- **Seeds:** Random state 42 used for all splits and model training.

## ML-12 — Demo and Summary

### 5-Minute Demo Outline
1. **The Problem (1m):** Explain the cost of stale content and the limits of simple 6-month rules.
2. **The Data (1m):** Show the 79M row warehouse and the feature engineering process.
3. **The Methodology (1m):** Explain the Grouped Client Split and why we must audit for site-specific signatures.
4. **The Results (1m):** Present the ROC-AUC comparison (0.772 vs 0.609) and what it means for generalization.
5. **The Playbook (1m):** Show the prioritized queue and the reason codes for human review.

### Social Post Cut
🚀 Can ML prioritize SEO content refreshes? We audited a model trained on 79M search records. The finding: a standard split suggests high performance (0.77 AUC), but generalization to new sites is harder (0.61 AUC). Validating for site-specific signatures is the difference between a dashboard that looks good and one that works. #SEO #MachineLearning #DataScience

### Employer-Facing Summary
I developed a content decay prediction system using a 79M-row search intelligence warehouse, transitioning from static rules to a data-driven prioritization queue. By implementing a rigorous client-holdout validation design, I quantified the generalization gap and established a decision-support framework that ranks high-value refresh opportunities for human review. This project demonstrates my ability to handle production-scale datasets while maintaining high standards for validation integrity and honest claim language.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.